# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vagisha14/Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I chose Logistic Regression as the Week 5 model because it is a simple and interpretable classification method. It provides a useful comparison with the Week 4 baseline while avoiding unnecessary model complexity. The model can also help identify which features are associated with the prediction through its coefficients.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I use a client-grouped train/test split so that rows from the same client do not appear in both training and test sets. This makes the evaluation more honest because the model is tested on clients it did not see during training. The split is consistent with the Week 4 baseline so the model comparison uses the same evaluation setup.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
import os
from pathlib import Path

repo_path = Path("/content/Internship")

if not repo_path.exists():
    !git clone https://github.com/Vagisha14/Internship.git /content/Internship

os.chdir("/content/Internship")

print("Working directory:", os.getcwd())
print("Dataset exists:",
      Path("data/raw/content_refresh_anonymized.csv").exists())

Cloning into '/content/Internship'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 164 (delta 58), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 2.02 MiB | 9.46 MiB/s, done.
Resolving deltas: 100% (58/58), done.
Working directory: /content/Internship
Dataset exists: True


In [7]:
# Section 3 — Train + compare vs Week 4 baseline

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ---------------------------
# 1. Load the same Week 4 data
# ---------------------------
data_path = Path("/content/Internship/data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

# ---------------------------
# 2. Define target
# ---------------------------
target = "is_declining_label"

# Create target if it is not already present
if target not in df.columns:
    if "trend_direction" in df.columns:
        df[target] = (
            df["trend_direction"]
            .astype(str)
            .str.lower()
            .isin(["declining", "decline", "down"])
            .astype(int)
        )
    else:
        raise ValueError("Target column could not be created.")

# Remove rows where target is missing
df = df.dropna(subset=[target]).copy()

y = df[target].astype(int)

# ---------------------------
# 3. Same client-grouped split
# ---------------------------
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, y, groups=groups)
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

# ---------------------------
# 4. Select safe features
# ---------------------------
exclude = {
    target,
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
}

features = [
    c for c in df.columns
    if c not in exclude
]

X_train = train[features]
X_test = test[features]

y_train = train[target].astype(int)
y_test = test[target].astype(int)

# Separate numeric and categorical columns
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

# ---------------------------
# 5. Preprocessing
# ---------------------------
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# ---------------------------
# 6. Logistic Regression model
# ---------------------------
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# ---------------------------
# 7. Predictions
# ---------------------------
pred = model.predict(X_test)

# ---------------------------
# 8. Evaluation
# ---------------------------
accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred, zero_division=0)
f1 = f1_score(y_test, pred, zero_division=0)

results = pd.DataFrame({
    "Model": ["Week 5 Logistic Regression"],
    "Accuracy": [round(accuracy, 4)],
    "Precision": [round(precision, 4)],
    "Recall": [round(recall, 4)],
    "F1": [round(f1, 4)]
})

print("\nWeek 5 model results:")
display(results)

# ---------------------------
# 9. Save comparison table
# ---------------------------
out_path = Path("work/outputs/week5_model_comparison.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

results.to_csv(out_path, index=False)

print("\nSaved:", out_path)

Dataset shape: (30000, 44)
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7

Week 5 model results:


,Model,Accuracy,Precision,Recall,F1
0,Week 5 Logistic Regression,0.7405,0.7183,0.8098,0.7613



Saved: work/outputs/week5_model_comparison.csv


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The Logistic Regression model can make errors when the available content and performance signals do not clearly separate declining and non-declining items. False positives may occur when an item looks weak based on its features but is not actually declining, while false negatives may occur when a decline is not strongly reflected in the available features. These errors show that the model should be used as decision support rather than as a final decision-maker.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.